# CNN Training

A convolutional network on log-mel spectrograms. This is the one model that does
not use the 215 summary features: it reads the spectrogram built by
`feature_extraction/spectrogram.py`, so it needs no scaler.

Run the cells in order. Use a GPU: Runtime > Change runtime type > T4 GPU.

In [ ]:
# Colab already ships TensorFlow, so only the audio and scikit-learn pins are
# installed here. The versions match requirements.txt.
!pip install -q scikit-learn==1.9.1 librosa==1.0.0 soundfile==0.14.0 joblib==1.6.0

print()
print("If Colab shows a 'Restart runtime' button above, click it, then carry on")
print("from the Drive mount cell. Do not run this install cell again.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:

PROJECT_PATH = '/content/drive/MyDrive/SonicSentinel'

In [ ]:
# A GPU makes a large difference here. Without one this notebook still runs, but
# the tuning pass will be slow.
import subprocess

result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
USE_GPU = result.returncode == 0 and "GPU" in result.stdout

if USE_GPU:
    print("GPU found:", result.stdout.strip())
else:
    print("No GPU found. Training on the CPU will be slow.")
    print("Set Runtime > Change runtime type > T4 GPU, then re-run from the mount cell.")

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("devices seen by TensorFlow:", tf.config.list_physical_devices())

In [ ]:
import os
import sys

if not os.path.isdir(PROJECT_PATH):
    raise RuntimeError(
        f"PROJECT_PATH does not exist: {PROJECT_PATH}\n"
        "Check the folder name in your Drive and fix PROJECT_PATH in the cell above."
    )

if not os.path.isfile(os.path.join(PROJECT_PATH, 'feature_extraction', 'spectrogram.py')):
    raise RuntimeError(
        f"{PROJECT_PATH} does not look like the project folder.\n"
        "It must contain app.py, config/, feature_extraction/ and src/."
    )

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# The spectrogram is built by the project's own module, imported from Drive. It
# is never redefined here: a second definition would train the network on
# spectrograms the application never produces.
try:
    import feature_extraction.spectrogram as spectrogram_module
    from feature_extraction.spectrogram import INPUT_SHAPE
    from config.config import CLASSES, PYTHON_MODELS
    from src import cnn_training
except Exception as error:
    raise RuntimeError(
        "Could not import the project modules from PROJECT_PATH.\n"
        "Check that the whole project folder was uploaded to Drive, including "
        "config/, feature_extraction/, audio_preprocessing/ and src/."
    ) from error

if not os.path.abspath(spectrogram_module.__file__).startswith(os.path.abspath(PROJECT_PATH)):
    raise RuntimeError(
        f"spectrogram.py was imported from {spectrogram_module.__file__}, "
        f"not from {PROJECT_PATH}. Fix sys.path before training."
    )

print("imported the project modules from", PROJECT_PATH)
print("spectrogram module:", spectrogram_module.__file__)
print("CNN input shape:", INPUT_SHAPE)
print("classes:", len(CLASSES))

In [ ]:
MODEL_NAME = "cnn"

# The tuning pass samples this many combinations from cnn_training.SEARCH_SPACE
# and scores each one on the validation split. Raise it if you have time.
CANDIDATES = 8

# Epochs per candidate during tuning, then for the final longer run. Both use
# early stopping, so these are upper limits rather than fixed counts.
TUNING_EPOCHS = 18
FINAL_EPOCHS = 60

print("search space:", cnn_training.SEARCH_SPACE)
print("combinations in total:", cnn_training.count_space())
print("candidates to be tried:", CANDIDATES)

In [ ]:
# Prints the split sizes first, then builds the spectrograms, tunes on the
# validation split, retrains the best settings for longer, and scores the test
# split once at the end.
#
# Spectrograms are cached to PROJECT_PATH/data/metadata/*_spectrograms.npz on
# Drive. If the runtime disconnects, run this cell again and it reuses the cache
# instead of rebuilding them.
model = cnn_training.run_training(
    candidates=CANDIDATES,
    tuning_epochs=TUNING_EPOCHS,
    final_epochs=FINAL_EPOCHS,
)

In [ ]:
# What this notebook wrote to Drive.
import os

for folder, names in [
    ('python_models', [PYTHON_MODELS[MODEL_NAME]['model_file'], 'label_encoder.pkl']),
    ('reports', [PYTHON_MODELS[MODEL_NAME]['metrics_file']]),
]:
    for name in names:
        path = os.path.join(PROJECT_PATH, folder, name)
        state = 'saved' if os.path.exists(path) else 'MISSING'
        print(f"{state:8} {folder}/{name}")

print()
print("Download those files into the same folders in the project. The CNN needs")
print("no scaler. To make it the active model, set ACTIVE_PYTHON_MODEL = 'cnn'")
print("in config/config.py.")